In [1]:
# ============================================================
# SHAP Analysis - Imports
# ============================================================

import os
import joblib
import shap
import pickle
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# ============================================================
# Verify Explainability Folders
# ============================================================

from pathlib import Path

EXPLAIN_DIR = Path("../explainability")
DEPENDENCE_DIR = EXPLAIN_DIR / "dependence_plots"

print("Explainability Folder Exists :", EXPLAIN_DIR.exists())
print("Dependence Folder Exists     :", DEPENDENCE_DIR.exists())

Explainability Folder Exists : True
Dependence Folder Exists     : True


In [9]:
# ============================================================
# Load Saved Models
# ============================================================

MODEL_DIR = "../models"

xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb.pkl"))

preprocessor = joblib.load(os.path.join(MODEL_DIR, "preprocessor.pkl"))

feature_names = joblib.load(os.path.join(MODEL_DIR, "feature_names.pkl"))

print("Models loaded successfully.\n")

print(type(xgb_model))
print(type(preprocessor))

print(f"\nNumber of transformed features : {len(feature_names)}")

Models loaded successfully.

<class 'xgboost.sklearn.XGBClassifier'>
<class 'sklearn.compose._column_transformer.ColumnTransformer'>

Number of transformed features : 24


In [10]:
# ============================================================
# Load Dataset
# ============================================================

DATA_PATH = "../data/train.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")

print(df.shape)

df.head()

Dataset loaded successfully.
(700000, 26)


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [12]:
# ============================================================
# Prepare Data
# ============================================================

TARGET_COLUMN = "diagnosed_diabetes"

X = df.drop(columns=[TARGET_COLUMN])

X_transformed = preprocessor.transform(X)

print("Original Shape :", X.shape)
print("Transformed Shape :", X_transformed.shape)

Original Shape : (700000, 25)
Transformed Shape : (700000, 42)


In [13]:
# ============================================================
# Create Sample for SHAP
# ============================================================

SAMPLE_SIZE = 2000
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

sample_indices = np.random.choice(
    X_transformed.shape[0],
    SAMPLE_SIZE,
    replace=False
)

X_shap = X_transformed[sample_indices]

print("SHAP Sample Shape :", X_shap.shape)

SHAP Sample Shape : (2000, 42)


In [16]:
# ============================================================
# Create SHAP Explainer
# ============================================================

print("Creating SHAP Explainer...")

explainer = shap.Explainer(
    xgb_model.predict,
    X_shap,
    feature_names=feature_names
)

print("SHAP Explainer created successfully.")

Creating SHAP Explainer...
SHAP Explainer created successfully.


In [17]:
# ============================================================
# Compute SHAP Values
# ============================================================

print("Computing SHAP values...")

shap_values = explainer(X_shap)

print("Done!")

print(shap_values.values.shape)

Computing SHAP values...


PermutationExplainer explainer: 2001it [01:11, 26.17it/s]                          

Done!
(2000, 42)


In [19]:
# ============================================================
# Regenerate Feature Names
# ============================================================

# Get transformed feature names directly from the preprocessor
feature_names = preprocessor.get_feature_names_out().tolist()

print("=" * 60)
print("FEATURE NAME VERIFICATION")
print("=" * 60)

print(f"Number of transformed features : {X_transformed.shape[1]}")
print(f"Number of feature names        : {len(feature_names)}")

assert len(feature_names) == X_transformed.shape[1], \
    "Mismatch between transformed features and feature names!"

print("\nFeature names regenerated successfully.")

FEATURE NAME VERIFICATION
Number of transformed features : 42
Number of feature names        : 42

Feature names regenerated successfully.


In [20]:
# ============================================================
# SHAP Summary Plot
# ============================================================

plt.figure(figsize=(12,8))

shap.summary_plot(
    shap_values.values,
    features=X_shap,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR / "shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("✓ shap_summary.png saved")

✓ shap_summary.png saved


In [21]:
# ============================================================
# SHAP Bar Plot
# ============================================================

plt.figure(figsize=(10,8))

shap.summary_plot(
    shap_values.values,
    features=X_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR / "shap_bar.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("✓ shap_bar.png saved")

✓ shap_bar.png saved


In [22]:
# ============================================================
# Feature Importance CSV
# ============================================================

importance = np.abs(shap_values.values).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP_Value": importance
})

importance_df = importance_df.sort_values(
    by="Mean_SHAP_Value",
    ascending=False
).reset_index(drop=True)

importance_df.to_csv(
    EXPLAIN_DIR / "feature_importance.csv",
    index=False
)

print("✓ feature_importance.csv saved")

display(importance_df.head(15))

✓ feature_importance.csv saved


,Feature,Mean_SHAP_Value
0,num__physical_activity_minutes_per_week,0.216232
1,num__age,0.110599
2,num__family_history_diabetes,0.059163
3,num__triglycerides,0.050374
4,num__bmi,0.033063
5,num__diet_score,0.029208
6,num__hdl_cholesterol,0.025944
7,num__ldl_cholesterol,0.021076
8,num__cholesterol_total,0.020288
9,num__heart_rate,0.019994


In [23]:
# ============================================================
# Generate Top-10 Dependence Plots
# ============================================================

top_features = importance_df["Feature"].head(10).tolist()

print("Generating dependence plots...\n")

for feature in top_features:

    plt.figure(figsize=(8,6))

    shap.dependence_plot(
        feature,
        shap_values.values,
        X_shap,
        feature_names=feature_names,
        show=False
    )

    plt.tight_layout()

    filename = feature.replace("/", "_").replace(" ", "_")

    plt.savefig(
        DEPENDENCE_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"✓ Saved {filename}.png")

print("\nAll dependence plots generated successfully.")

Generating dependence plots...

✓ Saved num__physical_activity_minutes_per_week.png
✓ Saved num__age.png
✓ Saved num__family_history_diabetes.png
✓ Saved num__triglycerides.png
✓ Saved num__bmi.png
✓ Saved num__diet_score.png
✓ Saved num__hdl_cholesterol.png
✓ Saved num__ldl_cholesterol.png
✓ Saved num__cholesterol_total.png
✓ Saved num__heart_rate.png

All dependence plots generated successfully.


<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

In [24]:
# ============================================================
# Update feature_names.pkl
# ============================================================

import joblib
from pathlib import Path

MODEL_DIR = Path("../models")

# Get the correct transformed feature names
feature_names = preprocessor.get_feature_names_out().tolist()

# Save them
joblib.dump(feature_names, MODEL_DIR / "feature_names.pkl")

print("=" * 60)
print("feature_names.pkl updated successfully!")
print("=" * 60)
print(f"Number of feature names saved: {len(feature_names)}")
print(f"Saved to: {MODEL_DIR / 'feature_names.pkl'}")

feature_names.pkl updated successfully!
Number of feature names saved: 42
Saved to: ..\models\feature_names.pkl
